# HW3 — Two retrievers and a model: what RAG actually sends

**Name: Magzhan Zhidekhan**
**Student ID: S23067686**
**Group: 9**
**Repository: https://github.com/magzhanzhidekhan-ux/magzhanzhidekhan-hw3**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>I used ChatGPT (OpenAI) as an AI assistant while working on this assignment.
>
>ChatGPT helped me understand the BM25 and cosine similarity formulas, explain the Python code, and understand how BM25 and EmbeddingGemma retrievers work.
>
>It also helped me troubleshoot errors related to Hugging Face model access, understand RAG prompts and token usage, and interpret the results of the Hit@3, LLM-only vs. RAG, and Top-K experiments.
>
>I used ChatGPT to help explain my results and prepare my written analysis. I ran the notebook, checked the outputs, and reviewed the conclusions myself.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = 8.3333

**Query terms:** how, does, rag, retrieve, information (5 terms, N = 3 documents)

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5)) = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln 7 = 1.9459 for every term.
Note: D2 contains "retrieves" and "retrieval", but no token is exactly "retrieve", so f = 0 for it.

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

> No term contributes: f(t, d) = 0 for every query term in every document, so the numerator idf · f · (k1 + 1) is 0 and each term is 0 / (positive number) = 0.
>
> D1: length normaliser = 1 − 0.75 + 0.75 · 9 / 8.3333 = 1.0600, denominator = 0 + 1.5 · 1.0600 = 1.5900.
> score = 1.9459 · 0 · 2.5 / 1.5900 (how) + (does) + (rag) + (retrieve) + (information) = 0 + 0 + 0 + 0 + 0 = 0
>
> D2: length normaliser = 1 − 0.75 + 0.75 · 10 / 8.3333 = 1.1500, denominator = 0 + 1.5 · 1.1500 = 1.7250.
> score = 0 + 0 + 0 + 0 + 0 = 0
>
> D3: length normaliser = 1 − 0.75 + 0.75 · 6 / 8.3333 = 0.7900, denominator = 0 + 1.5 · 0.7900 = 1.1850.
> score = 0 + 0 + 0 + 0 + 0 = 0
>
> All three documents score 0, so BM25 as written cannot rank them: it is a three-way tie.

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 1 (tie) |
| D2 | 0.0000 | 1 (tie) |
| D3 | 0.0000 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = 8.3333

**Stemmed query terms:** how, doe, rag, retriev, inform (N = 3 documents)

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

idf for the four absent terms = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln 7 = 1.9459.
idf(retriev) = ln((3 − 1 + 0.5) / (1 + 0.5)) = ln(2.5 / 1.5) = 0.5108.

**BM25 of each document**, worked:

> D2 is the only document in which a query term occurs. f(retriev, D2) = 2 (from "Retrieval" and "retrieves"), |D2| = 10.
> length normaliser = 1 − 0.75 + 0.75 · 10 / 8.3333 = 0.25 + 0.90 = 1.1500
> denominator = f + k1 · normaliser = 2 + 1.5 · 1.1500 = 3.7250
> numerator = idf · f · (k1 + 1) = 0.5108 · 2 · 2.5 = 2.5541
> term (retriev) = 2.5541 / 3.7250 = 0.6857
> The other four terms contribute 0 (f = 0).
> score(D2) = 0 + 0 + 0 + 0.6857 + 0 = 0.6857
>
> D1: no query term occurs (f = 0 for all five), so score(D1) = 0 + 0 + 0 + 0 + 0 = 0.
> D3: no query term occurs (f = 0 for all five), so score(D3) = 0 + 0 + 0 + 0 + 0 = 0.

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 2 (tie) |
| D2 | 0.6857 | 1 |
| D3 | 0.0000 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> In (a) BM25 retrieves no document first: all three scores are exactly 0, a three-way tie. BM25 only counts exact token matches, and none of the query tokens (how, does, rag, retrieve, information) appears literally in any document. D2 contains "retrieves" and "retrieval", which mean the same thing, but neither is the token "retrieve", so f = 0. That is not a calculation mistake: I rechecked the tokens and the zero holds, and it shows that BM25 cannot see that two word forms are related.
>
> In (b) BM25 retrieves D2 first, with 0.6857. Stemming turns "retrieve", "retrieves" and "retrieval" into the single token "retriev", so the query now matches D2, twice. Nothing else changed: D1 and D3 still share no stemmed token with the query, so they stay at 0 and tie for second. The score is also modest because "retriev" is in 1 of 3 documents (idf = 0.5108), and D2 is longer than the average (10 > 8.33), which lowers the term's weight slightly.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | √0.93 = 0.9644 | √(0.49+0.01+0.16) = √0.66 = 0.8124 | 0.78 / (0.9644 · 0.8124) = 0.9956 |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | √0.93 = 0.9644 | √(0.81+0.09+0.36) = √1.26 = 1.1225 | 1.08 / (0.9644 · 1.1225) = 0.9977 |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | √0.93 = 0.9644 | √(0.01+0.64+0.04) = √0.69 = 0.8307 | 0.34 / (0.9644 · 0.8307) = 0.4244 |

‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644.

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2, with cosine 0.9977. D1 is a very close second (0.9956), and the gap between them is only 0.0021, so on these vectors D1 and D2 are nearly tied. D3 is far behind (0.4244): its largest component is the second coordinate (0.8), while the query's largest is the first (0.8), so its direction points somewhere else.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Against A1 (a) there is nothing to compare: BM25 as written gave 0 to all three documents, a three-way tie, so it produced no ranking, while cosine produced a clear one.
> Against A1 (b) the top document agrees: both put D2 first. The rest does not agree. BM25 with stems scores D1 and D3 both at 0 and ties them for second place, whereas cosine puts D1 almost level with D2 (0.9956) and D3 far below (0.4244). So cosine separates D1 from D3 and BM25 cannot.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 only counts query tokens that appear in the document. The query is "How does RAG retrieve information?" and D1 is "Large language models can generate text and answer questions." They share no token at all, not even after stemming, so BM25 gives D1 exactly 0. Yet in the embedding space D1 is almost on top of the query (cosine 0.9956), because a text about language models answering questions is about the same topic as a question about how RAG retrieves information. The vectors capture that closeness of meaning, and exact word matching cannot.
> The opposite also holds: D3 ("Embeddings represent text as numerical vectors") shares no word with the query either, so BM25 gives it 0, and the vectors agree that it is far from the query (0.4244). On this tiny example BM25 and cosine agree about D3 but disagree about D1, and D1 is exactly the case where the document is relevant in meaning but has no word in common.
> One caveat: these vectors were given in the task, not produced by a real model, so this shows how the two methods can disagree, not that any particular model would place D1 there.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import math, re

docs = {
    "D1": "Large language models can generate text and answer questions.",
    "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
    "D3": "Embeddings represent text as numerical vectors.",
}
query = "How does RAG retrieve information?"
K1, B = 1.5, 0.75
STEMS = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
         "generate": "gener", "generating": "gener", "generation": "gener",
         "information": "inform", "does": "doe"}

def tokenize(text, stem=False):
    toks = re.findall(r"[a-z0-9]+", text.lower())
    return [STEMS.get(t, t) for t in toks] if stem else toks

def bm25_check(stem):
    D = {k: tokenize(v, stem) for k, v in docs.items()}
    q = tokenize(query, stem)
    N = len(D)
    avgdl = sum(len(t) for t in D.values()) / N
    print("lengths:", {k: len(v) for k, v in D.items()}, "| avgdl =", round(avgdl, 4))
    scores = {k: 0.0 for k in D}
    for t in q:
        n = sum(1 for toks in D.values() if t in toks)
        idf = math.log((N - n + 0.5) / (n + 0.5))
        tfs = {k: toks.count(t) for k, toks in D.items()}
        print(f"  {t:12} n={n}  idf={idf:.4f}  tf={tfs}")
        for k, toks in D.items():
            f = tfs[k]
            scores[k] += idf * f * (K1 + 1) / (f + K1 * (1 - B + B * len(toks) / avgdl))
    for k, s in sorted(scores.items(), key=lambda x: -x[1]):
        print("  ", k, round(s, 4))

print("--- A1 (a) as written ---")
bm25_check(False)
print("--- A1 (b) with stems ---")
bm25_check(True)

q = [0.8, 0.2, 0.5]
V = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
mag = lambda v: math.sqrt(sum(x * x for x in v))
dot = lambda v: sum(a * b for a, b in zip(q, v))
print("--- A2 cosine ---  |query| =", round(mag(q), 4))
for k, v in sorted(V.items(), key=lambda kv: -dot(kv[1]) / (mag(q) * mag(kv[1]))):
    print("  ", k, "dot =", round(dot(v), 4), "| |d| =", round(mag(v), 4),
          "| cos =", round(dot(v) / (mag(q) * mag(v)), 4))

--- A1 (a) as written ---
lengths: {'D1': 9, 'D2': 10, 'D3': 6} | avgdl = 8.3333
  how          n=0  idf=1.9459  tf={'D1': 0, 'D2': 0, 'D3': 0}
  does         n=0  idf=1.9459  tf={'D1': 0, 'D2': 0, 'D3': 0}
  rag          n=0  idf=1.9459  tf={'D1': 0, 'D2': 0, 'D3': 0}
  retrieve     n=0  idf=1.9459  tf={'D1': 0, 'D2': 0, 'D3': 0}
  information  n=0  idf=1.9459  tf={'D1': 0, 'D2': 0, 'D3': 0}
   D1 0.0
   D2 0.0
   D3 0.0
--- A1 (b) with stems ---
lengths: {'D1': 9, 'D2': 10, 'D3': 6} | avgdl = 8.3333
  how          n=0  idf=1.9459  tf={'D1': 0, 'D2': 0, 'D3': 0}
  doe          n=0  idf=1.9459  tf={'D1': 0, 'D2': 0, 'D3': 0}
  rag          n=0  idf=1.9459  tf={'D1': 0, 'D2': 0, 'D3': 0}
  retriev      n=1  idf=0.5108  tf={'D1': 0, 'D2': 2, 'D3': 0}
  inform       n=0  idf=1.9459  tf={'D1': 0, 'D2': 0, 'D3': 0}
   D2 0.6857
   D1 0.0
   D3 0.0
--- A2 cosine ---  |query| = 0.9644
   D2 dot = 1.08 | |d| = 1.1225 | cos = 0.9977
   D1 dot = 0.78 | |d| = 0.8124 | cos = 0.9956
   D3 dot = 0.3

## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>

In [3]:
import json, re
from rank_bm25 import BM25Okapi

corpus = [json.loads(l) for l in open("data/corpus.jsonl", encoding="utf-8")]
queries = json.load(open("data/queries.json", encoding="utf-8"))["queries"]

def tok(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tok(c["text"]) for c in corpus], k1=1.5, b=0.75)

def bm25_top(query, k=3):
    scores = bm25.get_scores(tok(query))
    order = sorted(range(len(corpus)), key=lambda i: -scores[i])[:k]
    return [(corpus[i]["id"], float(scores[i]), corpus[i]["text"].split("\n")[0][:80])
            for i in order]

bm25_hits = {}
for q in queries:
    top = bm25_top(q["text"])
    bm25_hits[q["id"]] = any(cid in q["relevant"] for cid, _, _ in top)
    print(f'\n{q["id"]}: {q["text"]}   (relevant: {q["relevant"]})')
    for rank, (cid, score, line) in enumerate(top, 1):
        print(f"  {rank}. {cid}  {score:7.3f}  {line}")
    print("  hit@3:", bm25_hits[q["id"]])

print("\nBM25 hit@3 on fixed queries:", sum(bm25_hits.values()), "/", len(bm25_hits))


Q-01: What is form GR-305 used for?   (relevant: ['GO-08', 'GO-12', 'GO-23'])
  1. GO-08   15.825  Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f
  2. GO-12    8.195  A student who disagrees with a decision may appeal within 14 calendar days of th
  3. GO-06    8.070  Form GR-104 is the application form for the need-based study grant. It is the on
  hit@3: True

Q-02: My family is poor. Is there financial help for students like me?   (relevant: ['GO-01', 'GO-02', 'GO-03'])
  1. KZ-17    8.726  The lake currently covers about 16,400 km2 (6,300 sq mi). However, like the Aral
  2. GO-20    8.311  The need-based grant is for full-time undergraduate students only. Part-time stu
  3. KZ-25    7.511  Beshbarmak is usually made from finely chopped boiled meat, mixed with dough (ty
  hit@3: False

Q-03: Өтінішті қашанға дейін тапсыру керек?   (relevant: ['GO-09', 'GO-14'])
  1. GO-14   10.698  2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін 

In [4]:
print(f"{'id':6}{'hit':7}top-3 (scores)")
for q in queries:
    top = bm25_top(q["text"])
    ids = ", ".join(f"{cid} ({s:.1f})" for cid, s, _ in top)
    print(f"{q['id']:6}{str(bm25_hits[q['id']]):7}{ids}   relevant={q['relevant']}")

id    hit    top-3 (scores)
Q-01  True   GO-08 (15.8), GO-12 (8.2), GO-06 (8.1)   relevant=['GO-08', 'GO-12', 'GO-23']
Q-02  False  KZ-17 (8.7), GO-20 (8.3), KZ-25 (7.5)   relevant=['GO-01', 'GO-02', 'GO-03']
Q-03  True   GO-14 (10.7), KZ-01 (0.0), KZ-02 (0.0)   relevant=['GO-09', 'GO-14']
Q-04  False  GO-05 (12.8), GO-17 (10.5), GO-19 (10.1)   relevant=['GO-02']
Q-05  True   KZ-05 (15.7), KZ-06 (6.4), KZ-11 (5.9)   relevant=['KZ-05']
Q-06  True   KZ-37 (19.1), KZ-38 (17.2), KZ-03 (9.4)   relevant=['KZ-37', 'KZ-38']
Q-07  False  GO-21 (14.7), GO-16 (9.2), GO-01 (9.1)   relevant=['GO-15']
Q-08  True   GO-17 (17.2), GO-05 (12.8), GO-04 (11.9)   relevant=['GO-17']


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [7]:
import os
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv(override=True)

model = SentenceTransformer(
    "google/embeddinggemma-300m",
    token=os.environ["HF_TOKEN"],
    device="cpu"
)

texts = [chunk["text"] for chunk in corpus]

document_embeddings = model.encode_document(
    texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

def embedding_top(query, k=3):
    query_embedding = model.encode_query(
        [query],
        convert_to_tensor=True
    )

    scores = model.similarity(
        query_embedding,
        document_embeddings
    )[0].cpu().numpy()

    order = np.argsort(scores)[::-1][:k]

    return [
        (
            corpus[i]["id"],
            float(scores[i]),
            corpus[i]["text"].split("\n")[0][:80]
        )
        for i in order
    ]

embedding_hits = {}

for q in queries:
    top = embedding_top(q["text"])

    embedding_hits[q["id"]] = any(
        chunk_id in q["relevant"]
        for chunk_id, _, _ in top
    )

    print(f"\n{q['id']}: {q['text']}")

    for rank, (chunk_id, score, first_line) in enumerate(top, 1):
        print(f"{rank}. {chunk_id} | {score:.4f} | {first_line}")

    print("Hit@3:", embedding_hits[q["id"]])

print(
    "\nEmbedding hit@3:",
    sum(embedding_hits.values()),
    "/",
    len(embedding_hits)
)

c:\Users\Asus\magzhanzhidekhan-hw3\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--google--embeddinggemma-300m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Asus\magzhanzhidekhan-hw3\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserW


Q-01: What is form GR-305 used for?
1. GO-08 | 0.6955 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f
2. GO-12 | 0.4609 | A student who disagrees with a decision may appeal within 14 calendar days of th
3. GO-23 | 0.4505 | Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в
Hit@3: True

Q-02: My family is poor. Is there financial help for students like me?
1. GO-01 | 0.4446 | The need-based study grant 2026 is run by the Grant Office for full-time undergr
2. GO-10 | 0.4117 | Proof of income must accompany form GR-210. Accepted: a salary certificate from 
3. GO-20 | 0.4090 | The need-based grant is for full-time undergraduate students only. Part-time stu
Hit@3: True

Q-03: Өтінішті қашанға дейін тапсыру керек?
1. GO-14 | 0.3992 | 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін студенттік по
2. GO-09 | 0.3502 | Applications for the 2026 grant open on 1 September 2026 and close on 15 October
3. GO-11 | 0.3

## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which document explains the GR-305 appeal form?| GO-08| BM25: the exact identifier GR-305 should help lexical matching.|
| Q-10 | How can students challenge a decision made by the Grant Office?| GO-08, GO-12| Embeddings: semantic matching can connect "challenge a decision" with "appeal".|

In [ ]:
my_queries = [
    {
        "id": "Q-09",
        "text": "Which document explains the GR-305 appeal form?",
        "relevant": ["GO-08"]
    },
    {
        "id": "Q-10",
        "text": "How can students challenge a decision made by the Grant Office?",
        "relevant": ["GO-08", "GO-12"]
    }
]

queries.extend(my_queries)

print("Total queries:", len(queries))

Total queries: 10


## Results for all ten or more queries: top 3 from each retriever

In [ ]:
bm25_hits = {}
embedding_hits = {}

for q in queries:
    qid = q["id"]

    bm25_results = bm25_top(q["text"], k=3)
    embedding_results = embedding_top(q["text"], k=3)

    bm25_ids = [item[0] for item in bm25_results]
    embedding_ids = [item[0] for item in embedding_results]

    bm25_hits[qid] = any(
        cid in q["relevant"] for cid in bm25_ids
    )

    embedding_hits[qid] = any(
        cid in q["relevant"] for cid in embedding_ids
    )

    print(f"\n{qid}: {q['text']}")

    print("BM25:")
    for rank, (cid, score, first_line) in enumerate(bm25_results, 1):
        print(f"  {rank}. {cid} | {score:.4f} | {first_line}")

    print("BM25 Hit@3:", bm25_hits[qid])

    print("Embedding:")
    for rank, (cid, score, first_line) in enumerate(embedding_results, 1):
        print(f"  {rank}. {cid} | {score:.4f} | {first_line}")

    print("Embedding Hit@3:", embedding_hits[qid])

print("\nBM25 total:", sum(bm25_hits.values()), "/", len(queries))
print("Embedding total:", sum(embedding_hits.values()), "/", len(queries))


Q-01: What is form GR-305 used for?
BM25:
  1. GO-08 | 15.8255 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f
  2. GO-12 | 8.1953 | A student who disagrees with a decision may appeal within 14 calendar days of th
  3. GO-06 | 8.0700 | Form GR-104 is the application form for the need-based study grant. It is the on
BM25 Hit@3: True
Embedding:
  1. GO-08 | 0.6955 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f
  2. GO-12 | 0.4609 | A student who disagrees with a decision may appeal within 14 calendar days of th
  3. GO-23 | 0.4505 | Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в
Embedding Hit@3: True

Q-02: My family is poor. Is there financial help for students like me?
BM25:
  1. KZ-17 | 8.7262 | The lake currently covers about 16,400 km2 (6,300 sq mi). However, like the Aral
  2. GO-20 | 8.3105 | The need-based grant is for full-time undergraduate students only. Part-time stu
  3.

In [11]:
print("| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |")
print("|---|---|:---:|---|:---:|")

for q in queries:
    qid = q["id"]

    b_ids = [item[0] for item in bm25_top(q["text"], 3)]
    e_ids = [item[0] for item in embedding_top(q["text"], 3)]

    b_hit = "Yes" if bm25_hits[qid] else "No"
    e_hit = "Yes" if embedding_hits[qid] else "No"

    print(
        f"| {qid} | {', '.join(b_ids)} | {b_hit} | "
        f"{', '.join(e_ids)} | {e_hit} |"
    )

print(
    f"| **hit@3 total** | | **{sum(bm25_hits.values())}/{len(queries)}** "
    f"| | **{sum(embedding_hits.values())}/{len(queries)}** |"
)

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | Yes | GO-08, GO-12, GO-23 | Yes |
| Q-02 | KZ-17, GO-20, KZ-25 | No | GO-01, GO-10, GO-20 | Yes |
| Q-03 | GO-14, KZ-01, KZ-02 | Yes | GO-14, GO-09, GO-11 | Yes |
| Q-04 | GO-05, GO-17, GO-19 | No | GO-02, GO-05, GO-17 | Yes |
| Q-05 | KZ-05, KZ-06, KZ-11 | Yes | KZ-05, KZ-06, KZ-03 | Yes |
| Q-06 | KZ-37, KZ-38, KZ-03 | Yes | KZ-37, KZ-38, KZ-10 | Yes |
| Q-07 | GO-21, GO-16, GO-01 | No | GO-15, GO-18, GO-01 | Yes |
| Q-08 | GO-17, GO-05, GO-04 | Yes | GO-17, GO-18, GO-15 | Yes |
| Q-09 | GO-08, GO-12, GO-10 | Yes | GO-08, GO-12, GO-23 | Yes |
| Q-10 | GO-01, GO-21, GO-08 | Yes | GO-08, GO-21, GO-12 | Yes |
| **hit@3 total** | | **7/10** | | **10/10** |


## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | Yes | GO-08, GO-12, GO-23 | Yes |
| Q-02 | KZ-17, GO-20, KZ-25 | No | GO-01, GO-10, GO-20 | Yes |
| Q-03 | GO-14, KZ-01, KZ-02 | Yes | GO-14, GO-09, GO-11 | Yes |
| Q-04 | GO-05, GO-17, GO-19 | No | GO-02, GO-05, GO-17 | Yes |
| Q-05 | KZ-05, KZ-06, KZ-11 | Yes | KZ-05, KZ-06, KZ-03 | Yes |
| Q-06 | KZ-37, KZ-38, KZ-03 | Yes | KZ-37, KZ-38, KZ-10 | Yes |
| Q-07 | GO-21, GO-16, GO-01 | No | GO-15, GO-18, GO-01 | Yes |
| Q-08 | GO-17, GO-05, GO-04 | Yes | GO-17, GO-18, GO-15 | Yes |
| Q-09 | GO-08, GO-12, GO-10 | Yes | GO-08, GO-12, GO-23 | Yes |
| Q-10 | GO-01, GO-21, GO-08 | Yes | GO-08, GO-21, GO-12 | Yes |
| **hit@3 total** | | **7/10** | | **10/10** |

**Both retrievers agree:** which query, and why was it easy?

>Both retrievers successfully found relevant documents for Q-01, Q-03, Q-05, Q-06, Q-08, Q-09, and Q-10. 
>
>For example, Q-01 was easy because the question contains the exact form code "GR-305". BM25 matched this code directly, while the embedding model also identified the relevant document GO-08.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

>BM25 did not outperform embedding retrieval on any of the ten queries in Hit@3.
>
>However, BM25 was effective for exact identifiers. For Q-01, it ranked GO-08 first with a score of 15.8255 because the document contains the exact form code "GR-305".

**Embeddings did better:** which query, and what could BM25 not match?

>Embedding retrieval performed better on Q-02, Q-04, and Q-07.
>
>For example, Q-02 asks about financial help for students from poor families. BM25 returned unrelated documents about a lake and Kazakh food, while embedding retrieval ranked GO-01, a document about need-based grants, first.
>
>This happened because embedding retrieval can match the meaning of a question even when the exact words are different.

In [12]:
print("BM25 vs Embedding Retrieval")
print("-" * 50)

for q in queries:
    qid = q["id"]

    bm25_result = bm25_hits[qid]
    embedding_result = embedding_hits[qid]

    if bm25_result and embedding_result:
        result = "Both correct"
    elif embedding_result:
        result = "Embedding better"
    elif bm25_result:
        result = "BM25 better"
    else:
        result = "Both missed"

    print(f"{qid}: {result}")

print("\nBM25 Hit@3:", sum(bm25_hits.values()), "/", len(queries))
print("Embedding Hit@3:", sum(embedding_hits.values()), "/", len(queries))

BM25 vs Embedding Retrieval
--------------------------------------------------
Q-01: Both correct
Q-02: Embedding better
Q-03: Both correct
Q-04: Embedding better
Q-05: Both correct
Q-06: Both correct
Q-07: Embedding better
Q-08: Both correct
Q-09: Both correct
Q-10: Both correct

BM25 Hit@3: 7 / 10
Embedding Hit@3: 10 / 10


---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [15]:
qa_questions = qa_data["questions"]

MODEL = "gpt-5.6-luna"

SYSTEM_PROMPT = (
    "You are a helpful Grant Office assistant. "
    "Answer the user's question using only the provided context. "
    "If the context does not contain the answer, say that you do not know. "
    "Do not invent facts."
)

def build_rag_messages(question, retriever="embedding", k=3):
    if retriever == "bm25":
        results = bm25_top(question, k=k)
    elif retriever == "embedding":
        results = embedding_top(question, k=k)
    else:
        raise ValueError("Unknown retriever")

    context_blocks = []

    for chunk_id, score, _ in results:
        chunk = next(c for c in corpus if c["id"] == chunk_id)

        context_blocks.append(
            f"[Chunk ID: {chunk_id}]\n{chunk['text']}"
        )

    context = "\n\n".join(context_blocks)

    user_message = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]

    return messages, results, context

In [16]:
encoding = tiktoken.get_encoding("o200k_base")

question = qa_questions[0]["question"]

messages, retrieved, context = build_rag_messages(
    question,
    retriever="embedding",
    k=3
)

print("=" * 70)
print("QUESTION:", question)
print("=" * 70)

for message in messages:
    print(f"\nROLE: {message['role']}")
    print(message["content"])

system_tokens = len(encoding.encode(SYSTEM_PROMPT))
question_tokens = len(encoding.encode(question))
context_tokens = len(encoding.encode(context))
user_message_tokens = len(encoding.encode(messages[1]["content"]))

total_content_tokens = system_tokens + user_message_tokens

print("\n" + "=" * 70)
print("TOKEN COUNT")
print("=" * 70)

print("System tokens:", system_tokens)
print("Question tokens:", question_tokens)
print("Retrieved context tokens:", context_tokens)
print("User message tokens:", user_message_tokens)
print("Total content tokens:", total_content_tokens)

print("\nRETRIEVED CHUNKS")

for chunk_id, score, _ in retrieved:
    chunk = next(c for c in corpus if c["id"] == chunk_id)
    chunk_tokens = len(
        encoding.encode(f"[Chunk ID: {chunk_id}]\n{chunk['text']}")
    )

    print(
        f"{chunk_id} | score={score:.4f} | "
        f"tokens={chunk_tokens} | characters={len(chunk['text'])}"
    )

print("Number of retrieved chunks:", len(retrieved))
print("Total context characters:", len(context))

QUESTION: How much does a student in income band 2 receive, and when is it paid?

ROLE: system
You are a helpful Grant Office assistant. Answer the user's question using only the provided context. If the context does not contain the answer, say that you do not know. Do not invent facts.

ROLE: user
Context:
[Chunk ID: GO-04]
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[Chunk ID: GO-03]
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households 

## C1 — the complete prompt, printed before it is sent

In [17]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print("MODEL ANSWER:")
print(response.choices[0].message.content)

print("\nTOKEN USAGE:")
print("API prompt tokens:", response.usage.prompt_tokens)
print("API completion tokens:", response.usage.completion_tokens)
print("API total tokens:", response.usage.total_tokens)

print("\nLOCAL TOKEN COUNT:", total_content_tokens)
print("API PROMPT TOKENS:", response.usage.prompt_tokens)
print(
    "DIFFERENCE:",
    response.usage.prompt_tokens - total_content_tokens
)

MODEL ANSWER:
A student in income band 2 receives **150,000 tenge per academic year**, paid in **two equal instalments of 75,000 tenge**: the first in **October** and the second in **February**.

TOKEN USAGE:
API prompt tokens: 374
API completion tokens: 51
API total tokens: 425

LOCAL TOKEN COUNT: 364
API PROMPT TOKENS: 374
DIFFERENCE: 10


In [18]:
print("Question:", question_tokens)
print("Context:", context_tokens)
print("GO-04:", len(encoding.encode(
    "[Chunk ID: GO-04]\n" +
    next(c["text"] for c in corpus if c["id"] == "GO-04")
)))

Question: 18
Context: 303
GO-04: 98


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 39|
| Question | 18|
| Retrieved context | 303|
| **Total input** | 364|
| **`usage.prompt_tokens` from the response** | 374|

- Number of retrieved chunks: 3
- Tokens per chunk: GO-04 — use the measured value; GO-03 — 112; GO-15 — 93.
- Total context size (tokens / characters): context_tokens tokens / 1,187 characters.

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>My local token count was 364, while the API reported 374 prompt tokens. The difference is 10 tokens. This happens because the API also counts message formatting and role-related overhead that is not included when I tokenize only the message contents.

**What exactly does retrieval add to the model's input?**

>Retrieval adds three relevant document chunks to the user message, including their IDs and full text. For this question, the retrieved chunks were GO-04, GO-03, and GO-15. They provide the grant amount, income band information, and payment schedule. The model uses this context to answer the question based on the documents rather than relying only on its general knowledge.

## C2 — three systems, six questions

In [19]:
import time

def ask_llm(question, retriever=None, k=3):
    if retriever is None:
        current_messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful Grant Office assistant. "
                    "If you do not know the answer, say so. "
                    "Do not invent facts."
                )
            },
            {
                "role": "user",
                "content": question
            }
        ]
        retrieved_ids = []
    else:
        current_messages, results, _ = build_rag_messages(
            question,
            retriever=retriever,
            k=k
        )
        retrieved_ids = [item[0] for item in results]

    response = client.chat.completions.create(
        model=MODEL,
        messages=current_messages
    )

    return {
        "answer": response.choices[0].message.content,
        "retrieved_ids": retrieved_ids,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens
    }


c2_results = []

for q in qa_questions:
    print(f"\n{'=' * 70}")
    print(q["id"], q["question"])
    print("REFERENCE:", q["reference"])

    row = {
        "id": q["id"],
        "question": q["question"],
        "reference": q["reference"],
        "relevant": q["relevant"]
    }

    for system_name, retriever in [
        ("LLM only", None),
        ("BM25 RAG", "bm25"),
        ("Embedding RAG", "embedding")
    ]:
        result = ask_llm(q["question"], retriever)

        row[system_name] = result

        print(f"\n{system_name}")
        print("Retrieved:", result["retrieved_ids"])
        print("Answer:", result["answer"])
        print("Prompt tokens:", result["prompt_tokens"])

    c2_results.append(row)

print("\nCompleted:", len(c2_results), "questions")


C-01 How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

LLM only
Retrieved: []
Answer: Which grant or program are you referring to? “Income band 2” varies by scheme, so I need the program or institution name to give the correct amount and payment date.
Prompt tokens: 52

BM25 RAG
Retrieved: ['GO-04', 'GO-02', 'GO-01']
Answer: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: the first in **October** and the second in **February**.
Prompt tokens: 377

Embedding RAG
Retrieved: ['GO-04', 'GO-03', 'GO-15']
Answer: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and the other in **February**.
Prompt tokens: 374

C-02 My grant application was refused. How long do I have to appeal, and which form do I u

In [20]:
for row in c2_results:
    print(f"\n{'=' * 60}")
    print(row["id"])
    print("QUESTION:", row["question"])
    print("REFERENCE:", row["reference"])

    for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
        result = row[system]

        print(f"\n{system}:")
        print("Retrieved:", result["retrieved_ids"])
        print("Answer:", result["answer"])


C-01
QUESTION: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

LLM only:
Retrieved: []
Answer: Which grant or program are you referring to? “Income band 2” varies by scheme, so I need the program or institution name to give the correct amount and payment date.

BM25 RAG:
Retrieved: ['GO-04', 'GO-02', 'GO-01']
Answer: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: the first in **October** and the second in **February**.

Embedding RAG:
Retrieved: ['GO-04', 'GO-03', 'GO-15']
Answer: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and the other in **February**.

C-02
QUESTION: My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days f

In [21]:
for row in c2_results:
    print(
        row["id"],
        "|",
        row["question"]
    )


C-01 | How much does a student in income band 2 receive, and when is it paid?
C-02 | My grant application was refused. How long do I have to appeal, and which form do I use?
C-03 | What is the minimum GPA needed to qualify for the need-based grant?
C-04 | When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
C-05 | When was Narxoz University founded, and what was it called at the start?
C-06 | I am a master's student. How much need-based grant can I get?


In [22]:
with open("c2_answers.txt", "w", encoding="utf-8") as f:
    for row in c2_results:
        f.write(f"\n{'=' * 60}\n")
        f.write(f"{row['id']}\n")
        f.write(f"REFERENCE: {row['reference']}\n")

        for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
            f.write(f"\n{system}:\n")
            f.write(row[system]["answer"] + "\n")

print("Saved to c2_answers.txt")

Saved to c2_answers.txt


| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ Asked for program details | ✓ 150,000 KZT, Oct/Feb | ✓ 150,000 KZT, Oct/Feb |
| C-02 | ✗ Did not specify deadline/form | ✓ 14 days, GR-305 | ✓ 14 days, GR-305 |
| C-03 | ✗ Did not specify GPA | ✓ GPA 2.67 | ✓ GPA 2.67 |
| C-04 | ✓ October 2017, 2031 | ✓ October 2017, 2031 | ✓ October 2017, 2031 |
| C-05 | ✓ 1963, Institute of National Economy | ✓ 1963, original name | ✓ 1963, original name |
| C-06 | ✗ Generic funding advice | ✓ Amount unknown, separate scheme | ✓ No amount given, separate scheme |
| **agrees with `reference`** | **2/6** | **6/6** | **6/6** |

**C2 — Comparison of Three Systems**

The LLM-only system agreed with the reference answers for 2 out of 6 questions (33.3%). Both BM25 RAG and Embedding RAG agreed with all 6 reference answers (100%).

For C-01, the LLM-only system could not identify the grant amount or payment dates. Both RAG systems correctly answered that students in income band 2 receive 150,000 tenge in two instalments, in October and February.

For C-03, the LLM-only system could not identify the minimum GPA. Both RAG systems correctly returned 2.67 instead of the outdated 3.0 requirement.

For C-06, the corpus does not specify the grant amount for master's students. Both RAG systems correctly avoided inventing an amount and referred to the separate graduate scheme.

These results show that retrieval improved answer accuracy by providing relevant information from the corpus. Although Embedding Retrieval had a higher Hit@3 score in Part B, both RAG systems achieved the same answer accuracy in Part C2.

## C3 — question 1 of 3: C-01

**1. What BM25 retrieved** (top 3, ids and scores):

- GO-04 — 20.2192
- GO-02 — 18.7422
- GO-01 — 17.2430

>BM25 retrieved GO-04 first, which directly contains the grant amount and payment schedule.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

- GO-04 — 0.5845
- GO-03 — 0.4842
- GO-15 — 0.4506

>EmbeddingGemma also retrieved GO-04 first. GO-03 explains the income bands, and GO-15 explains the payment schedule in Russian.

**3. What was sent to the model** (the context blocks):

>BM25 RAG received the full text of GO-04, GO-02, and GO-01. GO-04 explains the grant amount and payment dates, GO-02 describes eligibility requirements, and GO-01 introduces the need-based grant.
>
>Embedding RAG received the full text of GO-04, GO-03, and GO-15. GO-04 contains the grant amount, GO-03 defines income bands, and GO-15 confirms that payments are made in October and February.
>
>The context was included in the user message with each document's chunk ID.

**4. How retrieval changed the answer:**

>The LLM-only system could not answer the question and asked which grant program the user meant.
>
>Both RAG systems answered correctly: a student in income band 2 receives 150,000 tenge per academic year, paid in two instalments of 75,000 tenge in October and February.
>
>Retrieval made the answer more accurate, specific, and grounded in the provided documents.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence.**

>Neither RAG system missed the information required to answer this question.
>
>GO-04 states: "A student in band 2 receives 150,000 tenge for the academic year."
>
>It also confirms that payments are made in October and February.
>
>Both retrievers ranked GO-04 first, and both generated the correct answer. Therefore, there was no retrieval or generation failure in this case.
>
>The LLM-only system lacked the relevant context, which explains why it could not provide the specific grant amount and payment schedule.

In [23]:
for qid in ["C-01", "C-03", "C-06"]:
    row = next(r for r in c2_results if r["id"] == qid)

    print("\n" + "=" * 70)
    print(qid, row["question"])
    print("REFERENCE:", row["reference"])

    for system, retriever in [
        ("BM25 RAG", "bm25"),
        ("Embedding RAG", "embedding")
    ]:
        results = (
            bm25_top(row["question"], 3)
            if retriever == "bm25"
            else embedding_top(row["question"], 3)
        )

        print(f"\n{system}")

        for chunk_id, score, _ in results:
            chunk = next(c for c in corpus if c["id"] == chunk_id)

            print(f"\nChunk: {chunk_id}, Score: {score:.4f}")
            print("Context:", chunk["text"])

        print("\nAnswer:", row[system]["answer"])

    print("\nLLM-only answer:", row["LLM only"]["answer"])


C-01 How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

BM25 RAG

Chunk: GO-04, Score: 20.2192
Context: The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

Chunk: GO-02, Score: 18.7422
Context: An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enou

In [24]:
row = next(r for r in c2_results if r["id"] == "C-01")

for retriever in ["bm25", "embedding"]:
    messages, results, context = build_rag_messages(
        row["question"],
        retriever=retriever,
        k=3
    )

    print(f"\n=== {retriever.upper()} CONTEXT ===")
    print(context)


=== BM25 CONTEXT ===
[Chunk ID: GO-04]
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[Chunk ID: GO-02]
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

[Chunk ID: GO-01]
The need-based study grant 2026 is run by the Grant Office for full-time undergra

## C3 — question 2 of 3: C-03

**1. What BM25 retrieved (top 3, ids and scores):**

- GO-16 — 15.7985
- GO-02 — 14.0587
- GO-06 — 13.8516

>BM25 ranked GO-16 first, although it discusses scholarship combinations rather than the minimum GPA. However, it retrieved GO-02 in second place, which contains the correct 2026 GPA requirement.

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

- GO-02 — 0.6286
- GO-20 — 0.5691
- GO-05 — 0.5451

>EmbeddingGemma ranked GO-02 first, which contains the correct GPA requirement. It also retrieved GO-05, an archived document containing the outdated GPA requirement of 3.0.

**3. What was sent to the model (the context blocks):**

>BM25 RAG received GO-16, GO-02, and GO-06. These documents discuss scholarship combinations, the current 2026 eligibility rules, and the grant application form.
>
>Embedding RAG received GO-02, GO-20, and GO-05. These documents include the current eligibility rules, information about eligible students, and an archived 2025 policy.
>
>Both systems received the full text of their three retrieved chunks, including chunk IDs.
>
>The most important evidence was in GO-02: "their cumulative GPA is at least 2.67 on the 4.0 scale".
>
>GO-05 contains an outdated rule: "Under the 2025 scheme the minimum GPA for the grant was 3.0".

**4. How retrieval changed the answer:**

>The LLM-only system could not provide a specific GPA requirement and asked which grant program the user meant.
>
>Both BM25 RAG and Embedding RAG correctly answered that the minimum GPA is 2.67 on a 4.0 scale.
>
>Retrieval made the answer more specific and accurate. EmbeddingGemma also ranked the most relevant current document first.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence.**

>Neither RAG system produced an incorrect final answer.
>
>BM25 had a ranking problem because its first result, GO-16, was not directly relevant to the GPA question. However, GO-02 was retrieved in second place, so the correct information was available to the model.
>
>EmbeddingGemma retrieved both the current rule in GO-02 and the outdated rule in GO-05. This created a risk of using the wrong GPA.
>
>GO-05 explicitly says: "These rules were replaced on 1 June 2026".
>
>he model correctly used the current requirement of 2.67 instead of the archived requirement of 3.0.
>
>Therefore, there was no observed generation failure. BM25 showed a retrieval ranking weakness, while Embedding RAG demonstrated that the model could distinguish current information from outdated information.

In [25]:
row = next(r for r in c2_results if r["id"] == "C-03")

print("QUESTION:", row["question"])
print("REFERENCE:", row["reference"])

for name, retriever in [
    ("BM25", "bm25"),
    ("EmbeddingGemma", "embedding")
]:
    messages, results, context = build_rag_messages(
        row["question"],
        retriever=retriever,
        k=3
    )

    print(f"\n=== {name} ===")

    for chunk_id, score, _ in results:
        print(f"{chunk_id}: {score:.4f}")

    print("\nCONTEXT SENT:")
    print(context)

    print("\nANSWER:")
    print(row["BM25 RAG" if retriever == "bm25" else "Embedding RAG"]["answer"])

QUESTION: What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

=== BM25 ===
GO-16: 15.7985
GO-02: 14.0587
GO-06: 13.8516

CONTEXT SENT:
[Chunk ID: GO-16]
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[Chunk ID: GO-02]
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An app

## C3 — question 3 of 3: C-06

**1. What BM25 retrieved (top 3, ids and scores):**

- GO-16 — 16.9958
- GO-01 — 14.1725
- GO-20 — 13.7202

BM25 ranked GO-16 first, which discusses scholarship combinations and the 400,000 tenge support cap. GO-01 describes the undergraduate grant. The most relevant document, GO-20, appeared only in third place.

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

- GO-20 — 0.6664
- GO-16 — 0.6161
- GO-01 — 0.5539

EmbeddingGemma ranked GO-20 first. This was the most relevant document because it directly explains that master's students are covered by a separate scheme.

**3. What was sent to the model (the context blocks):**

BM25 RAG received the full text of GO-16, GO-01, and GO-20.

- GO-16 explains scholarship combinations and the maximum amount of university support.
- GO-01 describes the need-based grant for full-time undergraduate students.
- GO-20 explains that master's and doctoral students have a separate funding scheme.

Embedding RAG received the same three documents, but in a different order: GO-20, GO-16, and GO-01.

The most important evidence was in GO-20:

"Master's and doctoral students are supported by a separate scheme, which this handbook does not describe"

Neither retrieved context contained a specific grant amount for master's students.

**4. How retrieval changed the answer:**

The LLM-only system provided general information about graduate financial aid, including funding programs in the United States. However, it did not answer the question using the university's actual grant policy.

Both RAG systems provided more relevant and grounded answers.

BM25 RAG explicitly stated that it did not know the amount and explained that master's students must consult their department.

Embedding RAG explained that master's students cannot apply for the undergraduate need-based grant and should ask their department about the separate graduate scheme.

Retrieval helped both systems avoid inventing a grant amount that was not present in the corpus.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence.**

The exact grant amount for master's students was missing from the corpus itself.

GO-20 states: "they should ask their own department for its rules".

This is not a retrieval failure because both retrievers successfully found GO-20 within their top three results.

It is also not an observed generation failure because neither RAG system invented an amount.

However, BM25 had a ranking weakness: it placed GO-16 first, even though GO-20 was more directly relevant.

EmbeddingGemma performed better at ranking by placing GO-20 first.

This example demonstrates that a reliable RAG system should acknowledge missing information rather than generate an unsupported answer.

In [26]:
row = next(r for r in c2_results if r["id"] == "C-06")

print("QUESTION:", row["question"])
print("REFERENCE:", row["reference"])

for name, retriever in [
    ("BM25", "bm25"),
    ("EmbeddingGemma", "embedding")
]:
    messages, results, context = build_rag_messages(
        row["question"],
        retriever=retriever,
        k=3
    )

    print(f"\n=== {name} ===")

    for chunk_id, score, _ in results:
        print(f"{chunk_id}: {score:.4f}")

    print("\nCONTEXT SENT:")
    print(context)

    print("\nANSWER:")
    print(row["BM25 RAG" if retriever == "bm25" else "Embedding RAG"]["answer"])

QUESTION: I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.

=== BM25 ===
GO-16: 16.9958
GO-01: 14.1725
GO-20: 13.7202

CONTEXT SENT:
[Chunk ID: GO-16]
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[Chunk ID: GO-01]
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back un

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

>BM25 uses keyword matching and word frequency to rank documents. Embedding retrieval converts text into numerical vectors and compares their semantic similarity. BM25 focuses on exact words, while embeddings focus on meaning.

**2. When can BM25 perform better than embeddings?**

>BM25 can perform better when a question contains exact keywords, document IDs, or form numbers, such as GR-305. Exact matches can be more important than semantic similarity in these cases.

**3. When can embeddings perform better than BM25?**

>Embeddings perform better when the question and document use different words with similar meanings. In Part B, EmbeddingGemma achieved 10/10 Hit@3, while BM25 achieved 7/10. This shows that semantic retrieval worked better on our test queries.

**4. Why is cosine similarity used with embeddings?**

>Cosine similarity measures the angle between two vectors. It helps compare their directions rather than their lengths. Vectors pointing in similar directions usually represent semantically related texts.

**5. What information is actually sent to an LLM in a RAG system?**

>The LLM receives a system prompt, the user's question, and the text of the retrieved document chunks. In our experiment, each chunk also included its ID. The model does not receive the entire document collection.

**6. Does the LLM search the original documents itself?**

>No. The retriever searches the document collection and selects relevant chunks. These chunks are added to the prompt before the LLM generates an answer.

**7. What happens if retrieval returns an irrelevant chunk?**

>An irrelevant chunk may confuse the LLM, waste tokens, or cause an incorrect answer. However, the model can still answer correctly if other retrieved chunks contain the necessary information. In C-03, BM25 ranked GO-16 first, although GO-02 contained the correct GPA requirement.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>Increasing Top-K can improve RAG because more retrieved chunks may provide useful information. However, a larger Top-K can also introduce irrelevant or outdated documents, increase token usage, and potentially confuse the model.
>
>I tested Embedding RAG on question C-03: "What is the minimum GPA needed to qualify for the need-based grant?"

| Top-K | Retrieved chunks | Answer | Prompt tokens | Correct? |
|---|---|:---:|---|:---:|
| 1 | GO-02 | GPA 2.67 | 177 | ✓ |
| 2 | GO-02, GO-20, GO-05 | GPA 2.67 | 336 | ✓ |
| 3 | GO-02, GO-20, GO-05, GO-16, GO-01, GO-06, GO-19, GO-17 | GPA 2.67 | 792 | ✓ |


>All three configurations produced the correct answer: a minimum GPA of 2.67 on a 4.0 scale.
>
>With K=1, the model received only GO-02, which contained the correct information. With K=3, the context also included GO-05, an archived document with the outdated GPA requirement of 3.0. However, the model still selected >the current requirement correctly.
>
>With K=8, the answer remained correct, but prompt token usage increased to 792 compared with only 177 for K=1.
>
>Conclusion: Increasing Top-K did not improve accuracy for this question, but it increased token usage significantly. K=1 was the most efficient configuration in this experiment. A larger Top-K may help when important >evidence is missing from the first result, but it can also introduce unnecessary or conflicting information.

In [27]:
question = next(
    q["question"] for q in qa_questions
    if q["id"] == "C-03"
)

reference = next(
    q["reference"] for q in qa_questions
    if q["id"] == "C-03"
)

topk_results = []

for k in [1, 3, 8]:
    result = ask_llm(
        question,
        retriever="embedding",
        k=k
    )

    topk_results.append({
        "k": k,
        "ids": result["retrieved_ids"],
        "answer": result["answer"],
        "prompt_tokens": result["prompt_tokens"]
    })

print("QUESTION:", question)
print("REFERENCE:", reference)

for result in topk_results:
    print("\n" + "=" * 60)
    print("TOP-K:", result["k"])
    print("RETRIEVED:", result["ids"])
    print("ANSWER:", result["answer"])
    print("PROMPT TOKENS:", result["prompt_tokens"])

QUESTION: What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

TOP-K: 1
RETRIEVED: ['GO-02']
ANSWER: The minimum GPA needed is **2.67 on a 4.0 scale**.
PROMPT TOKENS: 177

TOP-K: 3
RETRIEVED: ['GO-02', 'GO-20', 'GO-05']
ANSWER: The minimum GPA is **2.67 on a 4.0 scale**.
PROMPT TOKENS: 336

TOP-K: 8
RETRIEVED: ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']
ANSWER: The minimum GPA is **2.67 on a 4.0 scale**.
PROMPT TOKENS: 792


**9. Is a vector database required to build a RAG system?**

>No, a vector database is not required to build a RAG system. RAG only needs a way to retrieve relevant information and send it to the LLM.
In our experiment, we used BM25 for keyword retrieval and SentenceTransformer embeddings with cosine similarity for semantic retrieval. The embeddings were stored and compared directly in Python without a vector database.
A vector database can be useful for large datasets because it provides efficient similarity search, indexing, and storage. However, for a small corpus like ours, it is not necessary.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>Embedding retrieval worked better overall for this corpus.
>
>In Part B, BM25 achieved a Hit@3 score of 7/10 (70%), while EmbeddingGemma achieved 10/10 (100%). This means EmbeddingGemma retrieved at least one relevant document in the top three results for every test query.
>
>In Part C2, both BM25 RAG and Embedding RAG agreed with the reference answers on all six questions (6/6, or 100%).
>
>Although both approaches produced equally accurate final answers in C2, EmbeddingGemma showed better retrieval performance.
>
>For example, in C-06, EmbeddingGemma ranked GO-20 first, while BM25 ranked it third. GO-20 was the most relevant document because it explained that master's students have a separate grant scheme.
>
>Therefore, EmbeddingGemma was more reliable at retrieving relevant documents for this corpus, while BM25 remained useful for exact keyword matching.

---
# Conclusion

>

In this assignment, I built and compared two retrieval approaches for a Retrieval-Augmented Generation (RAG) system: BM25 and EmbeddingGemma.

First, I learned how keyword-based retrieval and semantic retrieval work. BM25 ranks documents using word matching and frequency, while EmbeddingGemma represents text as numerical vectors and compares their semantic similarity.

n Part B, I tested both retrievers on ten questions. BM25 achieved 7/10 Hit@3 (70%), while EmbeddingGemma achieved 10/10 (100%). This showed that semantic retrieval performed better for our corpus.

In Part C, I connected both retrievers to the gpt-5.6-luna model. The LLM-only system agreed with 2 out of 6 reference answers, while both BM25 RAG and Embedding RAG agreed with all 6. This demonstrated how retrieved context can improve the accuracy and specificity of LLM answers.

I also analyzed retrieval and generation behavior. In C-03, the embedding retriever returned an outdated document, but the model correctly used the current GPA requirement. In C-06, both RAG systems avoided inventing a grant amount that was not provided in the corpus.

Finally, I tested Top-K values of 1, 3, and 8. All three produced the correct answer, but prompt token usage increased from 177 to 792. This showed that providing more context does not always improve answer quality and can increase costs.

Overall, I learned that RAG performance depends on both retrieval quality and the model's ability to use the retrieved evidence correctly. Embedding retrieval performed better in our retrieval tests, but both approaches were effective at generating grounded answers.